# Bibliotecas e configurações

In [1]:
import sys
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
from functools import lru_cache
from io import BytesIO
from pathlib import Path
from pyproj import Transformer

warnings.filterwarnings(
    "ignore",
    message=r"A NumPy version .* is required for this version of SciPy .*",
    category=UserWarning,
    module=r"dask\.array\.chunk_types",
)

In [2]:
def encontrar_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "src").exists() and (candidate / "src" / "config.py").exists():
            return candidate
    return cwd

In [3]:
ROOT = encontrar_root()
SRC_DIR = ROOT / "src"
INDEX_DIR = ROOT / "data" / "geoparquet"
ICECHUNK_DIR = ROOT / "data" / "icechunk_repo"


if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

In [4]:
import geopandas as gpd
import ipyleaflet
import ipywidgets as widgets
import matplotlib.pyplot as plt
import pandas as pd
import pystac_client
import rasterio
import rasterio.warp
import time
import traceback
import xarray as xr
import zarr
from IPython.display import HTML, clear_output, display
from storage import abrir_repositorio

zarr.config.set({"async.concurrency": 1})

from config import STAC_URL

TAMANHO_TILE_PADRAO = 512

print("Projeto:", ROOT)
print("GeoParquet dir:", INDEX_DIR)
print("Icechunk:", ICECHUNK_DIR)

Projeto: C:\Users\giuli\OneDrive\Área de Trabalho\INPE\bd. geo\trab_final_bdgeo\trab_final_bdgeo
GeoParquet dir: C:\Users\giuli\OneDrive\Área de Trabalho\INPE\bd. geo\trab_final_bdgeo\trab_final_bdgeo\data\geoparquet
Icechunk: C:\Users\giuli\OneDrive\Área de Trabalho\INPE\bd. geo\trab_final_bdgeo\trab_final_bdgeo\data\icechunk_repo


In [5]:
CACHE_CUBOS = {}
CACHE_SERIES = {}
CACHE_PIXELS_CUBO = {}
PAUSA_ENTRE_CHUNKS = 0.1

repo = abrir_repositorio(ICECHUNK_DIR)
sessao_icechunk = repo.readonly_session("main")
DS_ICECHUNK = xr.open_zarr(sessao_icechunk.store, consolidated=False)
VARIAVEIS_ICECHUNK = tuple(DS_ICECHUNK.data_vars)

ponto = {"lat": -23.1896, "lon": -45.8841}


[Sucesso] Repositório aberto: C:\Users\giuli\OneDrive\Área de Trabalho\INPE\bd. geo\trab_final_bdgeo\trab_final_bdgeo\data\icechunk_repo


In [6]:
cache = {}
CACHE_VALOR_RASTER = {}
MAX_RASTER_WORKERS = 4
ponto = {"lat": -23.1896, "lon": -45.8841}

LIMITES_SJC = ((-23.36, -46.12), (-22.92, -45.60))

## Coleções


In [7]:
def obter_cubo_virtual(colecao, variavel, inicio=None, fim=None):
    if variavel not in DS_ICECHUNK.data_vars:
        raise ValueError(f"Variável {variavel} não está disponível no Icechunk.")
    if colecao != "S2-16D-2":
        raise ValueError(f"Coleção {colecao} não está disponível no Icechunk.")

    chave = (colecao, variavel)
    ds = CACHE_CUBOS.get(chave, DS_ICECHUNK)
    da = ds[variavel]
    if "x" not in ds.coords or "y" not in ds.coords:
        tiepoint = da.attrs.get("model_tiepoint")
        pixel_scale = da.attrs.get("model_pixel_scale")
        if not tiepoint or not pixel_scale:
            raise ValueError("O cubo Icechunk não possui georreferenciamento persistido.")
        x = (tiepoint[3] + (pd.RangeIndex(ds.sizes["x"]).to_numpy() + 0.5) * pixel_scale[0])
        y = (tiepoint[4] - (pd.RangeIndex(ds.sizes["y"]).to_numpy() + 0.5) * pixel_scale[1])
        ds = ds.assign_coords(x=x, y=y)

    if not ds.attrs.get("crs"):
        href_col = f"{variavel}_href"
        gdf = pd.read_parquet(INDEX_DIR / f"{colecao}.parquet", columns=[href_col])
        href = gdf[href_col].dropna().iloc[0]
        with rasterio.open(href) as src:
            ds.attrs["crs"] = src.crs.to_wkt() if src.crs else None

    CACHE_CUBOS[chave] = ds
    return ds

In [8]:
def obter_crs_cubo(ds, variavel):
    da = ds[variavel]
    for attrs in [ds.attrs, da.attrs]:
        for chave in ["crs", "spatial_ref", "crs_wkt", "epsg"]:
            if chave in attrs and attrs[chave] is not None:
                return attrs[chave]

    for nome in ds.coords:
        for chave in ["spatial_ref", "crs_wkt", "crs", "proj4"]:
            if chave in ds[nome].attrs:
                return ds[nome].attrs[chave]

    if hasattr(da, "rio") and da.rio.crs is not None:
        return da.rio.crs

    return None

In [9]:
INDICES_POR_COLECAO = {"S2-16D-2": {variavel: variavel for variavel in sorted(VARIAVEIS_ICECHUNK)}}
COLECOES_DISPONIVEIS = sorted(INDICES_POR_COLECAO)
ROTULOS_VARIAVEIS = {}

if not VARIAVEIS_ICECHUNK:
    raise FileNotFoundError("Nenhuma variável encontrada no cubo Icechunk.")

print("Coleções disponíveis:", COLECOES_DISPONIVEIS)
print("Variáveis no Icechunk:", list(VARIAVEIS_ICECHUNK))

Coleções disponíveis: ['S2-16D-2']
Variáveis no Icechunk: ['B01', 'B02', 'B03', 'B04', 'NDVI']


## Conexão Icechunk


In [10]:
con = None

## Interface


### Mapa


In [11]:
mapa = ipyleaflet.Map(center=(ponto["lat"], ponto["lon"]), zoom=11, scroll_wheel_zoom=True)
mapa.fit_bounds(LIMITES_SJC)
mapa.layout = widgets.Layout(width="100%", height="500px", flex="1 1 560px")
marcador = ipyleaflet.Marker(location=(ponto["lat"], ponto["lon"]), draggable=True)
mapa.add_layer(marcador)

coordenadas = widgets.HTML()

### Coordenadas


In [12]:
def atualizar_coordenadas(lat, lon):
    ponto.update(lat=float(lat), lon=float(lon))
    coordenadas.value = (f"<b>Latitude:</b> {lat:.6f}&nbsp;&nbsp;&nbsp;<b>Longitude:</b> {lon:.6f}")

atualizar_coordenadas(**ponto)

### Eventos


In [13]:
def ao_clicar(**evento):
    if evento.get("type") == "click" and evento.get("coordinates"):
        lat, lon = evento["coordinates"]
        marcador.location = (lat, lon)
        atualizar_coordenadas(lat, lon)

def ao_arrastar(change):
    if change["name"] == "location":
        atualizar_coordenadas(*change["new"])

mapa.on_interaction(ao_clicar, remove=True)
mapa.on_interaction(ao_clicar)

try:
    marcador.unobserve(ao_arrastar, names="location")
except ValueError:
    pass

marcador.observe(ao_arrastar, names="location")

### Controles


In [14]:
colecao_widget = widgets.Dropdown(
    options=COLECOES_DISPONIVEIS,
    description="Coleção:",
    layout=widgets.Layout(width="100%"),
)

variavel_widget = widgets.Dropdown(description="Variável:", layout=widgets.Layout(width="100%"))

inicio_widget = widgets.DatePicker(description="Início:", value=(pd.Timestamp.today() - pd.DateOffset(months=6)).date())

fim_widget = widgets.DatePicker(description="Fim:", value=pd.Timestamp.today().date())
consultar = widgets.Button(
    description="Consultar série temporal",
    button_style="primary",
    icon="line-chart",
    layout=widgets.Layout(width="100%"),
)

limpar = widgets.Button(description="Limpar resultado", icon="trash", layout=widgets.Layout(width="100%"))

status = widgets.HTML()
saida = widgets.Output(layout=widgets.Layout(width="100%", overflow="hidden"))

### Funções auxiliares


In [15]:
def atualizar_variaveis(change=None):
    opcoes = [(ROTULOS_VARIAVEIS.get(variavel, variavel), variavel) for variavel in sorted(INDICES_POR_COLECAO.get(colecao_widget.value, {}))]
    variavel_widget.options = opcoes
    variavel_widget.value = opcoes[0][1] if opcoes else None
    consultar.disabled = not bool(opcoes)

try:
    colecao_widget.unobserve(atualizar_variaveis, names="value")
except ValueError:
    pass

colecao_widget.observe(atualizar_variaveis, names="value")
atualizar_variaveis()

In [16]:
def indice_selecionado():
    return INDICES_POR_COLECAO[colecao_widget.value][variavel_widget.value]

def card(titulo, valor):
    return widgets.HTML(f"""<div style='padding:12px 16px; min-width:110px; border:1px solid #dbeafe; border-radius:8px; background:#f8fafc'><div style='font-size:11px; color:#64748b'>{titulo}</div><div style='font-size:21px; font-weight:600; color:#0f172a'>{valor}</div></div>""")

def ler_valor_raster(linha, variavel, lon, lat):
    href = getattr(linha, f"{variavel}_href", None)
    if not href:
        return None

    chave = (href, variavel, round(lon, 6), round(lat, 6))
    if chave in CACHE_VALOR_RASTER:
        valor = CACHE_VALOR_RASTER[chave]
    else:
        try:
            with rasterio.open(href) as src:
                if src.crs is None:
                    return None
                
                if str(src.crs).upper() != "EPSG:4326":
                    x, y = rasterio.warp.transform("EPSG:4326", src.crs, [lon], [lat])
                    linha_pixel, coluna_pixel = src.index(x[0], y[0])
                else:
                    linha_pixel, coluna_pixel = src.index(lon, lat)

                if not (0 <= linha_pixel < src.height and 0 <= coluna_pixel < src.width):
                    return None
                
                dado = src.read(1, window=((linha_pixel, linha_pixel + 1), (coluna_pixel, coluna_pixel + 1),), masked=True,)
                valor = float(dado[0, 0]) if not dado.mask[0, 0] else float("nan")
                CACHE_VALOR_RASTER[chave] = valor
        except Exception:
            return None

    return pd.Timestamp(linha.datetime).tz_convert("UTC").tz_localize(None), valor


def extrair_valores_hrefs(gdf, variavel, lon, lat):
    registros = []
    with ThreadPoolExecutor(max_workers=MAX_RASTER_WORKERS) as executor:
        tarefas = [executor.submit(ler_valor_raster, linha, variavel, lon, lat) for linha in gdf.itertuples(index=False)]
        for tarefa in as_completed(tarefas):
            resultado = tarefa.result()
            if resultado is not None:
                registros.append(resultado)
                
    return (pd.Series(dict(registros)).sort_index() if registros else pd.Series(dtype=float))

In [17]:
def extrair_valores_cubo(ds, variavel, lon, lat):
    da = ds[variavel]
    if "x" not in ds.dims or "y" not in ds.dims:
        raise ValueError("O cubo não possui dimensões espaciais x/y.")

    if "x" not in ds.coords or "y" not in ds.coords:
        tiepoint = da.attrs.get("model_tiepoint")
        pixel_scale = da.attrs.get("model_pixel_scale")

        if not tiepoint or not pixel_scale:
            raise ValueError("O cubo não possui metadados para reconstruir x/y.")
        
        x_coords = (tiepoint[3] + (pd.RangeIndex(ds.sizes["x"]).to_numpy() + 0.5) * pixel_scale[0])
        y_coords = (tiepoint[4] - (pd.RangeIndex(ds.sizes["y"]).to_numpy() + 0.5) * pixel_scale[1])

        ds = ds.assign_coords(x=x_coords, y=y_coords)
        da = ds[variavel]

    crs_cubo = obter_crs_cubo(ds, variavel)

    if crs_cubo is None:
        raise ValueError("O cubo não possui CRS persistido.")

    transformer = Transformer.from_crs("EPSG:4326", str(crs_cubo), always_xy=True)
    x, y = transformer.transform(float(lon), float(lat))

    indice_x = int(abs(ds.x.values - x).argmin())
    indice_y = int(abs(ds.y.values - y).argmin())

    chunks_x = da.chunksizes.get("x", (512,))
    chunks_y = da.chunksizes.get("y", (512,))

    tamanho_x = int(chunks_x[0] if isinstance(chunks_x, tuple) else chunks_x)
    tamanho_y = int(chunks_y[0] if isinstance(chunks_y, tuple) else chunks_y)

    inicio_x = (indice_x // tamanho_x) * tamanho_x
    inicio_y = (indice_y // tamanho_y) * tamanho_y

    fim_x = min(inicio_x + tamanho_x, ds.sizes["x"])
    fim_y = min(inicio_y + tamanho_y, ds.sizes["y"])

    bloco_virtual = da.isel(x=slice(inicio_x, fim_x), y=slice(inicio_y, fim_y))
    tempos = pd.to_datetime(ds.time.values).tz_localize(None)

    try:
        bloco = bloco_virtual.compute()
        valores = bloco.values[:, indice_y - inicio_y, indice_x - inicio_x]
        return pd.Series(valores, index=tempos, name=variavel).dropna()
    except Exception as exc:
        if "429" not in str(exc):
            raise

    valores = []
    for indice_tempo, tempo in enumerate(tempos):
        chave = (variavel, pd.Timestamp(tempo), indice_x, indice_y)
        if chave in CACHE_PIXELS_CUBO:
            valores.append(CACHE_PIXELS_CUBO[chave])
            continue

        for tentativa in range(5):
            try:
                bloco = da.isel(time=slice(indice_tempo, indice_tempo + 1), x=slice(inicio_x, fim_x), y=slice(inicio_y, fim_y),).compute()
                valor = float(bloco.values[0, indice_y - inicio_y, indice_x - inicio_x])
                CACHE_PIXELS_CUBO[chave] = valor
                valores.append(valor)
                break
            except Exception as erro:
                if "429" not in str(erro):
                    raise
                if tentativa == 4:
                    raise RuntimeError(f"O BDC manteve o rate limit para a data {tempo.date()}.") from erro
                time.sleep(2**tentativa)
                
        if indice_tempo < len(tempos) - 1:
            time.sleep(PAUSA_ENTRE_CHUNKS)

    return pd.Series(valores, index=tempos, name=variavel).dropna()

In [18]:
def consultar_serie(_):
    with saida:
        clear_output(wait=True)
        colecao = colecao_widget.value
        variavel = variavel_widget.value
        lat, lon = ponto["lat"], ponto["lon"]
        inicio, fim = inicio_widget.value, fim_widget.value

        if colecao is None or variavel is None:
            status.value = '<span style="color:#991b1b">Selecione uma coleção e uma variável.</span>'
            return
        if inicio is None or fim is None or inicio > fim:
            status.value = ('<span style="color:#991b1b">Escolha um período válido.</span>')
            return

        chave = (colecao, variavel, round(lat, 6), round(lon, 6), str(inicio), str(fim))
        status.value = '<span style="color:#075985">Consultando cubo virtual...</span>'

        try:
            if chave in CACHE_SERIES:
                serie, tempo_total = CACHE_SERIES[chave]
            else:
                t0 = time.perf_counter()
                ds = obter_cubo_virtual(colecao, variavel, inicio, fim)
                inicio_ts = pd.Timestamp(inicio)
                fim_ts = (pd.Timestamp(fim).normalize() + pd.Timedelta(1, unit="D") - pd.Timedelta(1, unit="us"))
                ds_periodo = ds.sel(time=slice(inicio_ts, fim_ts))
                serie = extrair_valores_cubo(ds_periodo, variavel, lon, lat)

                if serie.empty or serie.notna().sum() == 0:
                    status.value = '<span style="color:#92400e">Não há valores válidos para esta consulta.</span>'
                    return
                
                tempo_total = time.perf_counter() - t0
                CACHE_SERIES[chave] = (serie, tempo_total)

            validos = int(serie.notna().sum())
            display(
                widgets.HBox(
                    [
                        card("OBSERVAÇÕES", len(serie)),
                        card("VÁLIDOS", validos),
                        card("TEMPO", f"{tempo_total:.2f}s"),
                    ],
                    layout=widgets.Layout(flex_flow="row wrap", grid_gap="10px"),
                )
            )

            display(HTML(f"""<div style='margin:14px 0; padding:12px 15px; border:1px solid #e2e8f0; border-radius:8px'><b>Consulta realizada</b><br><br><b>Coordenadas:</b> {lat:.6f}, {lon:.6f}<br><b>Período:</b> {inicio} → {fim}<br><b>Variável:</b> {variavel}<br><b>Coleção:</b> {colecao}</div>"""))

            fig, ax = plt.subplots(figsize=(7.2, 4.4), dpi=100)
            ax.plot(
                serie.index,
                serie.values,
                color="#087e8b",
                linewidth=2.2,
                marker="o",
                markersize=4,
                markerfacecolor="white",
                markeredgewidth=1.5,
            )

            ax.set(title=f"Série temporal — {variavel}", xlabel="Data", ylabel=variavel)
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)
            ax.spines["left"].set_color("#cbd5e1")
            ax.spines["bottom"].set_color("#cbd5e1")

            ax.grid(axis="y", color="#cbd5e1", alpha=0.55)
            fig.tight_layout()
            buffer = BytesIO()

            fig.savefig(buffer, format="png", bbox_inches="tight")
            plt.close(fig)

            grafico = widgets.Image(value=buffer.getvalue(), format="png", layout=widgets.Layout(width="720px", max_width="100%"),)

            display(widgets.HBox([grafico], layout=widgets.Layout( width="100%", justify_content="center", overflow="hidden"),))
            display(HTML(f"""<div style='margin-top:12px; padding:12px 15px; border:1px solid #e2e8f0; border-radius:8px'><b>Estatísticas da série</b><br><br>Mínimo: {serie.min():.3f}&nbsp;&nbsp;|&nbsp;&nbsp;Máximo: {serie.max():.3f}&nbsp;&nbsp;|&nbsp;&nbsp;Média: {serie.mean():.3f}&nbsp;&nbsp;|&nbsp;&nbsp;Mediana: {serie.median():.3f}</div>"""))
            status.value = '<span style="color:#166534">Consulta concluída.</span>'
            
        except Exception as exc:
            traceback.print_exc()
            status.value = f'<span style="color:#991b1b">Erro: {exc}</span>'

In [19]:
def limpar_resultado(_):
    with saida:
        clear_output(wait=True)

    status.value = ""

## Dashboard


In [20]:
consultar._click_handlers.callbacks = []
consultar.on_click(consultar_serie)
limpar._click_handlers.callbacks = []
limpar.on_click(limpar_resultado)

In [21]:
cabecalho = widgets.HTML("""<div style='padding:18px 22px; margin-bottom:12px; border:1px solid #bae6fd; border-radius:10px; background:linear-gradient(135deg,#ecfeff,#f8fafc)'><h2 style='margin:0 0 5px'>Dashboard de Séries Temporais</h2><div style='color:#475569'>Consulta interativa de cubos de dados</div></div>""")
painel = widgets.VBox(
    [
        widgets.HTML('<h4 style="margin:0 0 8px">Ponto de consulta</h4>'),
        coordenadas,
        widgets.HTML(
            '<div style="margin:6px 0 14px; color:#64748b; font-size:12px">Clique no mapa ou arraste o marcador.</div>'
        ),
        widgets.HTML('<h4 style="margin:0 0 8px">Cubo de dados</h4>'),
        colecao_widget,
        variavel_widget,
        widgets.HTML('<h4 style="margin:16px 0 8px">Período</h4>'),
        inicio_widget,
        fim_widget,
        widgets.HTML('<div style="height:10px"></div>'),
        consultar,
        limpar,
        status,
    ],
    layout=widgets.Layout(
        width="360px",
        flex="0 1 360px",
        padding="18px",
        border="1px solid #e2e8f0",
        overflow="hidden",
    ),
)

In [22]:
clear_output(wait=True)
dashboard = widgets.VBox(
    [
        cabecalho,
        widgets.HBox(
            [mapa, painel,],
            layout=widgets.Layout(width="100%", flex_flow="row wrap", grid_gap="18px", overflow="hidden"),
        ),
        widgets.HTML('<h3 style="margin:20px 0 10px">Resultado</h3>'),
        saida,
    ],
    layout=widgets.Layout(width="100%", overflow="hidden"),
)

display(dashboard, display_id="dashboard-principal")

<DisplayHandle display_id=dashboard-principal>